In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()
train.info()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# original distribution
sns.histplot(train['SalePrice'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('SalePrice Distribution (Original)')

# log transformed
sns.histplot(np.log1p(train['SalePrice']), kde=True, ax=axes[1], color='green')
axes[1].set_title('SalePrice Distribution (Log Transformed)')

plt.tight_layout()
plt.show()

print(f"Original Skewness: {train['SalePrice'].skew():.4f}")
print(f"Log Transformed Skewness: {np.log1p(train['SalePrice']).skew():.4f}")

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0].head(20))

plt.figure(figsize=(12, 6))
missing_top = missing_df[missing_df['Missing Count'] > 0].head(20)
sns.barplot(x=missing_top['Missing %'], y=missing_top.index, 
            hue=missing_top.index, palette='coolwarm', legend=False)
plt.title('Top 20 Columns with Missing Values')
plt.xlabel('Missing %')
plt.show()

In [ ]:
corr = train.select_dtypes(include=[np.number]).corr()
top_corr = corr['SalePrice'].sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_corr.values, y=top_corr.index,
            hue=top_corr.index, palette='coolwarm', legend=False)
plt.title('Top 15 Features Correlated with SalePrice')
plt.xlabel('Correlation')
plt.show()

print(top_corr)
top_features = top_corr.index.tolist()[:11]

plt.figure(figsize=(12, 8))
sns.heatmap(train[top_features].corr(), 
            annot=True, fmt='.2f', 
            cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap — Top Features')
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# overall quality vs price
sns.boxplot(x='OverallQual', y='SalePrice', data=train, ax=axes[0][0], palette='coolwarm')
axes[0][0].set_title('Overall Quality vs Sale Price')

# living area vs price
axes[0][1].scatter(train['GrLivArea'], train['SalePrice'], alpha=0.5, color='blue')
axes[0][1].set_xlabel('GrLivArea')
axes[0][1].set_ylabel('SalePrice')
axes[0][1].set_title('Living Area vs Sale Price')

# garage area vs price
axes[1][0].scatter(train['GarageArea'], train['SalePrice'], alpha=0.5, color='green')
axes[1][0].set_xlabel('GarageArea')
axes[1][0].set_ylabel('SalePrice')
axes[1][0].set_title('Garage Area vs Sale Price')

# year built vs price
axes[1][1].scatter(train['YearBuilt'], train['SalePrice'], alpha=0.5, color='red')
axes[1][1].set_xlabel('YearBuilt')
axes[1][1].set_ylabel('SalePrice')
axes[1][1].set_title('Year Built vs Sale Price')

plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
neighborhood_price = train.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)
sns.barplot(x=neighborhood_price.index, y=neighborhood_price.values,
            hue=neighborhood_price.index, palette='viridis', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Median Sale Price by Neighborhood')
plt.ylabel('Median SalePrice')
plt.show()

In [ ]:
# two clear outliers visible in GrLivArea plot
# large area but very low price — likely special sales
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000))]

print("Shape after removing outliers:", train.shape)


In [ ]:
# save test IDs for submission
test_ids = test['Id']

# log transform target before separating
train['SalePrice'] = np.log1p(train['SalePrice'])
y = train['SalePrice']

# combine for consistent cleaning
all_data = pd.concat([train.drop('SalePrice', axis=1), test], axis=0)
all_data.drop('Id', axis=1, inplace=True)

print("Combined shape:", all_data.shape)
# columns where NaN actually means "None" / "No feature"
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
             'BsmtFinType2', 'MasVnrType', 'MSSubClass']

for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

# columns where NaN means 0
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
             'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
             'MasVnrArea']

for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# fill with mode (most common value)
mode_cols = ['MSZoning', 'Electrical', 'KitchenQual',
             'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional']

for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# LotFrontage — fill with median of same neighborhood
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Utilities — almost all same value, just drop it
all_data.drop(['Utilities'], axis=1, inplace=True)

print("Missing values remaining:", all_data.isnull().sum().sum())

In [ ]:
# 1. Total Square Footage — most important new feature
all_data['TotalSF'] = (all_data['TotalBsmtSF'] + 
                        all_data['1stFlrSF'] + 
                        all_data['2ndFlrSF'])

# 2. House Age and Remodel Age
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodelAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# 3. Total Bathrooms
all_data['TotalBath'] = (all_data['FullBath'] + 
                          all_data['BsmtFullBath'] + 
                          0.5 * all_data['HalfBath'] + 
                          0.5 * all_data['BsmtHalfBath'])

# 4. Has Pool, Has Garage, Has Basement, Has Fireplace
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)

# 5. Porch Total
all_data['TotalPorch'] = (all_data['OpenPorchSF'] + 
                           all_data['EnclosedPorch'] + 
                           all_data['3SsnPorch'] + 
                           all_data['ScreenPorch'])

# 6. Quality × Condition interaction
all_data['QualCond'] = all_data['OverallQual'] * all_data['OverallCond']

# 7. Quality × Living Area interaction
all_data['QualLivArea'] = all_data['OverallQual'] * all_data['GrLivArea']

print("New features created successfully")
print("Shape:", all_data.shape)

In [ ]:
# label encode ordinal features (have natural ordering)
from sklearn.preprocessing import LabelEncoder

ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual',
                'GarageCond', 'PoolQC']

quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

for col in ordinal_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].map(quality_map).fillna(0)

# one hot encode remaining categorical columns
all_data = pd.get_dummies(all_data)

print("Shape after encoding:", all_data.shape)
from scipy.stats import skew

# find numerical features
numeric_feats = all_data.dtypes[all_data.dtypes != 'object'].index

# calculate skewness
skewness = all_data[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed_feats = skewness[abs(skewness) > 0.75].index

print(f"Number of skewed features: {len(skewed_feats)}")

# log transform skewed features
all_data[skewed_feats] = np.log1p(all_data[skewed_feats])

print("Skewness fixed")

In [ ]:
# split back
X_train = all_data[:len(train)]
X_test = all_data[len(train):]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# scale features — important for linear models
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# cross validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model, X, y):
    rmse = np.sqrt(-cross_val_score(model, X, y, 
                                     scoring='neg_mean_squared_error', 
                                     cv=kf))
    return rmse

print("Setup complete")

In [ ]:
lr = LinearRegression()
lr_scores = rmse_cv(lr, X_train_scaled, y)

print("Linear Regression:")
print(f"  RMSE Mean: {lr_scores.mean():.4f}")
print(f"  RMSE Std:  {lr_scores.std():.4f}")

# Ridge adds L2 penalty to prevent overfitting
ridge = Ridge(alpha=10)
ridge_scores = rmse_cv(ridge, X_train_scaled, y)

print("Ridge Regression:")
print(f"  RMSE Mean: {ridge_scores.mean():.4f}")
print(f"  RMSE Std:  {ridge_scores.std():.4f}")

# Lasso adds L1 penalty — also does feature selection
lasso = Lasso(alpha=0.0005)
lasso_scores = rmse_cv(lasso, X_train_scaled, y)

print("Lasso Regression:")
print(f"  RMSE Mean: {lasso_scores.mean():.4f}")
print(f"  RMSE Std:  {lasso_scores.std():.4f}")


In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42
)
rf_scores = rmse_cv(rf, X_train, y)

print("Random Forest:")
print(f"  RMSE Mean: {rf_scores.mean():.4f}")
print(f"  RMSE Std:  {rf_scores.std():.4f}")

xgb = XGBRegressor(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_scores = rmse_cv(xgb, X_train, y)

print("XGBoost:")
print(f"  RMSE Mean: {xgb_scores.mean():.4f}")
print(f"  RMSE Std:  {xgb_scores.std():.4f}")

In [ ]:
model_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso', 
              'Random Forest', 'XGBoost'],
    'RMSE Mean': [lr_scores.mean(), ridge_scores.mean(), 
                  lasso_scores.mean(), rf_scores.mean(), 
                  xgb_scores.mean()],
    'RMSE Std': [lr_scores.std(), ridge_scores.std(), 
                 lasso_scores.std(), rf_scores.std(), 
                 xgb_scores.std()]
}).sort_values('RMSE Mean')

print(model_comparison.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(x='RMSE Mean', y='Model', 
            hue='Model', data=model_comparison, 
            palette='coolwarm', legend=False)
plt.title('Model RMSE Comparison (Lower is Better)')
plt.show()

In [ ]:
# base models
estimators = [
    ('ridge', Ridge(alpha=10)),
    ('lasso', Lasso(alpha=0.0005)),
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42))
]

# meta model
stack = StackingRegressor(
    estimators=estimators,
    final_estimator=XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    cv=5
)

stack_scores = rmse_cv(stack, X_train_scaled, y)
print("Stacking Ensemble:")
print(f"  RMSE Mean: {stack_scores.mean():.4f}")
print(f"  RMSE Std:  {stack_scores.std():.4f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y, test_size=0.2, random_state=42
)

# fit best models
ridge.fit(X_tr, y_tr)
lasso.fit(X_tr, y_tr)
xgb.fit(X_train, y)

# blend predictions
ridge_val = ridge.predict(X_val)
lasso_val = lasso.predict(X_val)

val_pred = (ridge_val * 0.4 + lasso_val * 0.6)

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
r2 = r2_score(y_val, val_pred)

print(f"Validation RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
residuals = y_val - val_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# residuals vs predicted
axes[0].scatter(val_pred, residuals, alpha=0.5, color='blue')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted')

# residual distribution
sns.histplot(residuals, kde=True, ax=axes[1], color='green')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print("Residuals should be:")
print("- Randomly scattered around 0 (no pattern)")
print("- Roughly normally distributed")


xgb.fit(X_train, y)

feat_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', 
            hue='Feature', data=feat_imp, 
            palette='viridis', legend=False)
plt.title('Top 20 Feature Importances — XGBoost')
plt.show()

In [ ]:
# fit on full training data
ridge.fit(X_train_scaled, y)
lasso.fit(X_train_scaled, y)
xgb.fit(X_train, y)

# blend predictions
ridge_pred = ridge.predict(X_test_scaled)
lasso_pred = lasso.predict(X_test_scaled)
xgb_pred = xgb.predict(X_test)

# weighted blend
final_pred = (ridge_pred * 0.25 + 
              lasso_pred * 0.35 + 
              xgb_pred * 0.40)


final_pred = np.expm1(final_pred)

print("Predictions generated")
print("Sample predictions:", final_pred[:5])
print("Min price:", final_pred.min())
print("Max price:", final_pred.max())




In [ ]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_pred
})

submission.to_csv('submission.csv', index=False)

print("Submission saved!")
print("Shape:", submission.shape)
print("\nFirst 5 rows:")
print(submission.head())

In [ ]:
verify = pd.read_csv('submission.csv')

print("Shape:", verify.shape)
print("Any missing:", verify.isnull().sum().sum())
print("Min SalePrice:", verify['SalePrice'].min())
print("Max SalePrice:", verify['SalePrice'].max())
print("Mean SalePrice:", verify['SalePrice'].mean())
verify.head()